# Notebook 6 - Electrostatic Force Impact on Profiles and Spatial Spectra

This notebook explores how the **sign of the DC electrostatic bias** changes the cantilever response in the **default Bradler-style case**.

We compare two strong electrostatic scenarios:
- Strong electrostatics with `Delta V = +8 V`
- Strong electrostatics with `Delta V = -8 V`

For each case we inspect:
1. High-resolution displacement and slope profiles at the dominant contact resonance
2. Full-length coarse plots over the entire cantilever
3. Zoomed plots around the key null regions
   - slope null near `x/L ~ 0.6`
   - displacement null near the tip (`x/L -> 1`)
4. Full and zoomed **2D spatial-frequency spectra** for both `|z(x,f)|` and `|z'(x,f)|`

The goal is to see how reversing the DC bias changes the mode shape, the null locations, and the spatial-frequency spectra.

This notebook now normalizes spatial position by the contact location `L1 = cant.contact_x`, so nonzero `cant.tip_offset` can be explored consistently.

In [ ]:
import os, sys
# Use the package from this repository even if it has not been pip-installed.
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
import os
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from eb_cr_afm import (
    default_params,
    frequency_sweep,
    sho_analysis,
    solve_single_frequency,
    compute_2d_spectra,
)
from eb_cr_afm.params import ExcitationParams, SolverOptions

cwd = os.getcwd()
if os.path.basename(cwd).lower() == 'examples':
    RESULTS_DIR = os.path.abspath(os.path.join(cwd, '../results'))
else:
    RESULTS_DIR = os.path.abspath(os.path.join(cwd, 'results'))
os.makedirs(RESULTS_DIR, exist_ok=True)

cant, contact, exc_base, opts_base = default_params()
opts_base.method = 'analytical'
opts_base.f_start = 300e3
opts_base.f_stop = 400e3
opts_base.f_step = 500.0

print(f'Cantilever length L = {cant.L*1e6:.1f} um')
print(f'Base contact stiffness k1 = {contact.k1:.1f} N/m')
print(f'Results directory: {os.path.abspath(RESULTS_DIR)}')

## 1 - Build the electrostatic comparison cases

The contact mechanics stay fixed. Only the sign of the DC electrostatic bias is changed.

- **Positive DC bias**: strong distributed electrostatic load with `Delta V = +8 V`
- **Negative DC bias**: same magnitude, but with `Delta V = -8 V`

This notebook uses both electrostatic channels:
- distributed electrostatic load `q(x)`
- a local tip force `f0` coupled to `V_AC * Delta V`

With this setup, changing the sign of `Delta V` flips the phase of both electrostatic contributions relative to the piezo drive, so the interference can become more constructive or more destructive depending on the CPD sign.

In [ ]:
cases = [
    (
        'Strong electrostatics (+8 V DC)',
        '+8 V DC',
        dict(use_distributed_load=True, use_f0=True, bias_coupled_f0=True, f0=100e-12, delta_V=8.0, V_AC=8.0),
        '#e45756',
    ),
    (
        'Strong electrostatics (-8 V DC)',
        '-8 V DC',
        dict(use_distributed_load=True, use_f0=True, bias_coupled_f0=True, f0=100e-12, delta_V=-8.0, V_AC=8.0),
        '#4c78a8',
    ),
]

def make_excitation(overrides):
    vals = exc_base.__dict__.copy()
    vals.update(overrides)
    return ExcitationParams(**vals)

profile_data = {}

for name, short_label, overrides, color in cases:
    exc = make_excitation(overrides)
    sweep = frequency_sweep(cant, contact, exc, opts_base)
    mode = sho_analysis(sweep)[0]
    f_res = mode['f_res']

    opts_profile = SolverOptions(
        n_points=2400,
        f_start=f_res - 1e3,
        f_stop=f_res + 1e3,
        f_step=250.0,
        g=opts_base.g,
        method='analytical',
    )
    res = solve_single_frequency(f_res, cant, contact, exc, opts_profile)

    xL = res['x'] / cant.L
    z_amp = np.abs(res['z'])
    zp_amp = np.abs(res['zp'])

    slope_mask = (xL >= 0.45) & (xL <= 0.70)
    disp_mask = (xL >= 0.92) & (xL <= 1.00)
    slope_idx = np.where(slope_mask)[0][np.argmin(zp_amp[slope_mask])]
    disp_idx = np.where(disp_mask)[0][np.argmin(z_amp[disp_mask])]

    profile_data[name] = {
        'color': color,
        'short_label': short_label,
        'exc': exc,
        'sweep': sweep,
        'mode': mode,
        'res': res,
        'xL': xL,
        'z_amp': z_amp,
        'zp_amp': zp_amp,
        'z_norm': z_amp / max(np.max(z_amp), 1e-30),
        'zp_norm': zp_amp / max(np.max(zp_amp), 1e-30),
        'peak_disp_pm': float(np.max(z_amp) * 1e12),
        'peak_slope': float(np.max(zp_amp)),
        'tip_disp_pm': float(z_amp[-1] * 1e12),
        'tip_slope': float(zp_amp[-1]),
        'slope_null_xL': float(xL[slope_idx]),
        'disp_null_xL': float(xL[disp_idx]),
    }

print('Dominant resonance / profile summary:')
for name, data in profile_data.items():
    print(
        f"{name:24s}  f_res = {data['mode']['f_res']/1e3:7.3f} kHz   "
        f"max |z| = {data['peak_disp_pm']:10.3f} pm   "
        f"max |z'| = {data['peak_slope']:10.3e}   "
        f"slope null x/L = {data['slope_null_xL']:.4f}   "
        f"disp null x/L = {data['disp_null_xL']:.4f}"
    )

In [ ]:
disp_zoom = (0.96, 1.00)
slope_zoom = (0.50, 0.66)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey='row')

for name, data in profile_data.items():
    color = data['color']
    xL = data['xL']

    axes[0, 0].plot(xL, data['z_norm'], lw=2.2, color=color, label=name)
    axes[0, 1].plot(xL, data['z_norm'], lw=2.2, color=color)
    axes[1, 0].plot(xL, data['zp_norm'], lw=2.2, color=color, label=name)
    axes[1, 1].plot(xL, data['zp_norm'], lw=2.2, color=color)

    axes[0, 1].scatter(data['disp_null_xL'], data['z_norm'][np.argmin(np.abs(xL - data['disp_null_xL']))],
                       color=color, s=40, zorder=5)
    axes[1, 1].scatter(data['slope_null_xL'], data['zp_norm'][np.argmin(np.abs(xL - data['slope_null_xL']))],
                       color=color, s=40, zorder=5)

axes[0, 0].set_title('Displacement profile |z(x)| / max|z|')
axes[0, 1].set_title('Near-tip displacement null')
axes[1, 0].set_title("Slope profile |z'(x)| / max|z'|")
axes[1, 1].set_title("Slope null near x/L ~ 0.6")

axes[0, 1].set_xlim(*disp_zoom)
axes[1, 1].set_xlim(*slope_zoom)

for ax in axes.flat:
    ax.grid(True, alpha=0.25)
    ax.set_xlabel('Normalized position x/L')

axes[0, 0].set_ylabel('|z| / max|z|')
axes[1, 0].set_ylabel("|z'| / max|z'|")

axes[0, 0].legend(loc='best')
axes[1, 0].legend(loc='best')

fig.suptitle('Impact of DC-bias sign on spatial profiles at the dominant contact resonance', y=1.02)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

for name, data in profile_data.items():
    color = data['color']
    xL = data['xL']

    axes[0].plot(xL, data['z_amp'] * 1e12, lw=2.2, color=color, label=name)
    axes[1].plot(xL, data['zp_amp'], lw=2.2, color=color, label=name)

axes[0].set_title('Displacement profile |z(x)| in physical units')
axes[1].set_title("Slope profile |z'(x)| in physical units")
axes[0].set_ylabel('|z| (pm)')
axes[1].set_ylabel("|z'| (dimensionless)")
axes[1].set_xlabel('Normalized position x/L')

for ax in axes:
    ax.grid(True, alpha=0.25)
    ax.legend(loc='best')

fig.suptitle('Absolute-amplitude spatial profiles at the dominant contact resonance', y=1.02)
fig.tight_layout()

## 2 - Full and zoomed spatial-frequency spectra

The 2D maps are computed for both strong electrostatic cases:
- `Delta V = +8 V`
- `Delta V = -8 V`

For each quantity we show:
- a full cantilever map over `x/L in [0, 1]`
- a high-resolution zoom around the null region

The displacement maps are zoomed near the tip, while the slope maps are zoomed around the internal null near `x/L ~ 0.6`.

In [ ]:
spectra_cases = ['Strong electrostatics (+8 V DC)', 'Strong electrostatics (-8 V DC)']

for name in spectra_cases:
    data = profile_data[name]
    f_res = data['mode']['f_res']
    opts_spectra = SolverOptions(
        n_points=300,
        f_start=20e3,
        f_stop=2200e3,
        f_step=250.0,
        g=opts_base.g,
        method='analytical',
    )
    data['spectra'] = compute_2d_spectra(
        cant,
        contact,
        data['exc'],
        opts_spectra,
        n_x_points=240,
        verbose=False,
    )

def plot_spatial_spectra_comparison(case_names, amp_key, title, zoom_xlim):
    fig = make_subplots(
        rows=2,
        cols=len(case_names),
        shared_yaxes=True,
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
        subplot_titles=[
            f"{profile_data[name]['short_label']} - full" for name in case_names
        ] + [
            f"{profile_data[name]['short_label']} - zoom" for name in case_names
        ],
    )

    for col, name in enumerate(case_names, start=1):
        data = profile_data[name]
        sp = data['spectra']
        xL = sp['x_grid'] / cant.L
        freqs_khz = sp['freqs'] / 1e3
        z_map = np.log10(np.clip(sp[amp_key], 1e-30, None)).T

        for row in (1, 2):
            fig.add_trace(
                go.Heatmap(
                    x=xL,
                    y=freqs_khz,
                    z=z_map,
                    colorscale='Plasma',
                    zmin=float(np.nanmax(z_map) - 8.0),
                    zmax=float(np.nanmax(z_map)),
                    colorbar=dict(title='log10 amplitude', len=0.36, x=1.02) if (row == 1 and col == len(case_names)) else None,
                    showscale=(row == 1 and col == len(case_names)),
                    hovertemplate='x/L = %{x:.4f}<br>f = %{y:.2f} kHz<br>log10 A = %{z:.3f}<extra></extra>',
                ),
                row=row,
                col=col,
            )

        fig.update_xaxes(title_text='x/L', row=1, col=col)
        fig.update_xaxes(title_text='x/L', row=2, col=col, range=list(zoom_xlim))
        fig.update_yaxes(title_text='Frequency (kHz)', row=1, col=col)
        fig.update_yaxes(title_text='Frequency (kHz)', row=2, col=col)

    fig.update_layout(height=760, template='plotly_white', title=title)
    return fig

fig_disp = plot_spatial_spectra_comparison(
    spectra_cases,
    amp_key='z_amp',
    title='Displacement spectra |z(x,f)|: +8 V DC vs -8 V DC',
    zoom_xlim=disp_zoom,
)
fig_disp.show()

fig_slope = plot_spatial_spectra_comparison(
    spectra_cases,
    amp_key='zp_amp',
    title="Slope spectra |z'(x,f)|: +8 V DC vs -8 V DC",
    zoom_xlim=slope_zoom,
)
fig_slope.show()

In [ ]:
plus_name = 'Strong electrostatics (+8 V DC)'
minus_name = 'Strong electrostatics (-8 V DC)'
plus_data = profile_data[plus_name]
minus_data = profile_data[minus_name]
modes_plus = sho_analysis(plus_data['sweep'])
modes_minus = sho_analysis(minus_data['sweep'])
sp_plus = plus_data['spectra']
sp_minus = minus_data['spectra']
xL = sp_plus['x_grid'] / cant.L
freqs_khz = sp_plus['freqs'] / 1e3

def find_antiresonance_frequency(sweep_result, modes):
    freqs = sweep_result['freqs']
    amps = sweep_result['amplitudes_slope']
    mode_freqs = sorted(m['f_res'] for m in modes)

    if len(mode_freqs) >= 2:
        f_lo, f_hi = mode_freqs[0], mode_freqs[1]
        mask = (freqs >= f_lo) & (freqs <= f_hi)
        if np.count_nonzero(mask) >= 3:
            local_idx = np.argmin(amps[mask])
            return float(freqs[mask][local_idx])

    return float(freqs[np.argmin(amps)])

def local_feature_map(log_map, x_mask):
    zoom_map = log_map[x_mask, :]
    column_center = np.nanmedian(zoom_map, axis=0, keepdims=True)
    return zoom_map - column_center

def trace_feature_position(feature_map, x_coords, which='dark'):
    selector = np.argmin if which == 'dark' else np.argmax
    idx = selector(feature_map, axis=0)
    return x_coords[idx]

def suppress_trace_outliers(trace, window=21, sigma_thresh=3.5):
    clean = np.array(trace, dtype=float, copy=True)
    n = clean.size
    half = max(window // 2, 1)
    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)
        neighborhood = clean[lo:hi]
        med = np.nanmedian(neighborhood)
        mad = np.nanmedian(np.abs(neighborhood - med))
        scale = 1.4826 * mad
        if scale < 1e-8:
            continue
        if abs(clean[i] - med) > sigma_thresh * scale:
            clean[i] = med
    return clean

def best_shift_per_frequency(ref_map, cmp_map, x_coords, max_shift_pts=18):
    dx = float(np.mean(np.diff(x_coords)))
    shifts = np.full(ref_map.shape[1], np.nan)
    corr_peaks = np.full(ref_map.shape[1], np.nan)
    lag_grid = np.arange(-max_shift_pts, max_shift_pts + 1)
    for i in range(ref_map.shape[1]):
        a = ref_map[:, i]
        b = cmp_map[:, i]
        best_corr = -np.inf
        best_lag = 0
        for lag in lag_grid:
            if lag < 0:
                aa = a[-lag:]
                bb = b[:lag]
            elif lag > 0:
                aa = a[:-lag]
                bb = b[lag:]
            else:
                aa = a
                bb = b
            denom = np.linalg.norm(aa) * np.linalg.norm(bb)
            if denom <= 0:
                continue
            corr = float(np.dot(aa, bb) / denom)
            if corr > best_corr:
                best_corr = corr
                best_lag = lag
        shifts[i] = best_lag * dx
        corr_peaks[i] = best_corr
    return shifts, corr_peaks

log_zp_plus = np.log10(np.clip(sp_plus['zp_amp'], 1e-30, None))
log_zp_minus = np.log10(np.clip(sp_minus['zp_amp'], 1e-30, None))
x_zoom_mask = (xL >= slope_zoom[0]) & (xL <= slope_zoom[1])
x_zoom = xL[x_zoom_mask]
zoom_plus_raw = log_zp_plus[x_zoom_mask, :]
zoom_minus_raw = log_zp_minus[x_zoom_mask, :]
zoom_plus_feat = local_feature_map(log_zp_plus, x_zoom_mask)
zoom_minus_feat = local_feature_map(log_zp_minus, x_zoom_mask)
zoom_delta_raw = zoom_plus_raw - zoom_minus_raw
delta_scale = float(np.nanpercentile(np.abs(zoom_delta_raw), 80))
delta_scale = max(delta_scale, 1e-6)
zoom_delta_emph = np.arcsinh(12.0 * zoom_delta_raw / delta_scale)
raw_limits = np.nanpercentile(np.concatenate([zoom_plus_raw.ravel(), zoom_minus_raw.ravel()]), [1.0, 99.8])
diff_lim = float(np.nanpercentile(np.abs(zoom_delta_emph), 99.5))
diff_lim = max(diff_lim, 1.0)

dark_plus = trace_feature_position(zoom_plus_feat, x_zoom, which='dark')
dark_minus = trace_feature_position(zoom_minus_feat, x_zoom, which='dark')
bright_plus = trace_feature_position(zoom_plus_feat, x_zoom, which='bright')
bright_minus = trace_feature_position(zoom_minus_feat, x_zoom, which='bright')
shift_xcorr, corr_peak = best_shift_per_frequency(zoom_plus_feat, zoom_minus_feat, x_zoom)
dark_plus = suppress_trace_outliers(dark_plus)
dark_minus = suppress_trace_outliers(dark_minus)
bright_plus = suppress_trace_outliers(bright_plus)
bright_minus = suppress_trace_outliers(bright_minus)
shift_xcorr = suppress_trace_outliers(shift_xcorr, window=31, sigma_thresh=4.0)

f_res_plus = float(modes_plus[0]['f_res'])
f_res_minus = float(modes_minus[0]['f_res'])
f_ar_plus = find_antiresonance_frequency(plus_data['sweep'], modes_plus)
f_ar_minus = find_antiresonance_frequency(minus_data['sweep'], modes_minus)

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey='row')

panel_specs = [
    (axes[0, 0], zoom_plus_raw.T, '+8 V DC: raw slope zoom', 'magma', raw_limits[0], raw_limits[1], 'log10 amplitude'),
    (axes[0, 1], zoom_minus_raw.T, '-8 V DC: raw slope zoom', 'magma', raw_limits[0], raw_limits[1], 'log10 amplitude'),
    (axes[0, 2], zoom_delta_emph.T, 'Contrast-amplified difference: (+8V) - (-8V)', 'RdBu_r', -diff_lim, diff_lim, 'asinh-amplified delta'),
]

for ax, z_map, title, cmap, vmin, vmax, cbar_label in panel_specs:
    im = ax.imshow(
        z_map,
        origin='lower',
        aspect='auto',
        extent=[x_zoom[0], x_zoom[-1], freqs_khz[0], freqs_khz[-1]],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(title)
    ax.set_xlabel('Normalized position x/L')
    ax.set_ylabel('Frequency (kHz)')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=cbar_label)

axes[1, 0].plot(dark_plus, freqs_khz, color='#d62728', lw=2, label='+8 V dark line')
axes[1, 0].plot(dark_minus, freqs_khz, color='#1f77b4', lw=2, label='-8 V dark line')
axes[1, 0].set_title('Tracked dark-line position vs frequency')
axes[1, 0].set_xlabel('Normalized position x/L')
axes[1, 0].set_ylabel('Frequency (kHz)')
axes[1, 0].grid(True, alpha=0.25)
axes[1, 0].legend(loc='best')

axes[1, 1].plot(bright_plus, freqs_khz, color='#ff7f0e', lw=2, label='+8 V bright line')
axes[1, 1].plot(bright_minus, freqs_khz, color='#2ca02c', lw=2, label='-8 V bright line')
axes[1, 1].set_title('Tracked bright-line position vs frequency')
axes[1, 1].set_xlabel('Normalized position x/L')
axes[1, 1].grid(True, alpha=0.25)
axes[1, 1].legend(loc='best')

axes[1, 2].plot(freqs_khz, 1e3 * (dark_plus - dark_minus), color='black', lw=2, label='Dark-line shift')
axes[1, 2].plot(freqs_khz, 1e3 * (bright_plus - bright_minus), color='#9467bd', lw=2, label='Bright-line shift')
axes[1, 2].plot(freqs_khz, 1e3 * shift_xcorr, color='#8c564b', lw=2, ls='--', label='Cross-corr shift')
axes[1, 2].axhline(0.0, color='0.5', ls='--', lw=1)
axes[1, 2].set_title('Bias-induced shift in zoomed spectral features')
axes[1, 2].set_xlabel('Frequency (kHz)')
axes[1, 2].set_ylabel('Shift in x/L (x10^-3)')
axes[1, 2].grid(True, alpha=0.25)
axes[1, 2].legend(loc='best')

fig.suptitle('DC-bias comparison in slope x-f space: raw zoom with contrast-amplified difference', y=1.02)
fig.tight_layout()

print(
    'Peak tracked shifts in the slope zoom window: '
    f"dark max = {1e3 * np.nanmax(np.abs(dark_plus - dark_minus)):.3f} x/L x10^-3, "
    f"bright max = {1e3 * np.nanmax(np.abs(bright_plus - bright_minus)):.3f} x/L x10^-3, "
    f"xcorr max = {1e3 * np.nanmax(np.abs(shift_xcorr)):.3f} x/L x10^-3"
)

## 3 - Resonance and antiresonance shape-difference maps

The spectra show where the bright and dark bands move, but they do not directly show **where along the beam** the `+8 V` and `-8 V` responses differ.

This section compares the spatial profiles at two special frequencies:
- **Resonance**: the first fitted contact-resonance peak
- **Antiresonance**: the minimum in tip slope amplitude between the first and second detected resonances, when a second resonance is present in the sweep window

For each frequency we overlay `|z(x)|` and `|z'(x)|` for `+8 V` and `-8 V`, then plot the pointwise differences. That makes it easier to see whether the bias mainly rescales the mode or changes the shape locally.

In [ ]:
def solve_profile_at_frequency(data, freq_hz, n_points=2600):
    opts_local = SolverOptions(
        n_points=n_points,
        f_start=freq_hz - 1e3,
        f_stop=freq_hz + 1e3,
        f_step=250.0,
        g=opts_base.g,
        method='analytical',
    )
    return solve_single_frequency(freq_hz, cant, contact, data['exc'], opts_local)

f_res_cmp = 0.5 * (f_res_plus + f_res_minus)
f_ar_cmp = 0.5 * (f_ar_plus + f_ar_minus)
f_ref = 10e3

res_plus_res = solve_profile_at_frequency(plus_data, f_res_cmp)
res_minus_res = solve_profile_at_frequency(minus_data, f_res_cmp)
res_plus_ar = solve_profile_at_frequency(plus_data, f_ar_cmp)
res_minus_ar = solve_profile_at_frequency(minus_data, f_ar_cmp)
res_plus_ref = solve_profile_at_frequency(plus_data, f_ref)
res_minus_ref = solve_profile_at_frequency(minus_data, f_ref)

def plot_shape_difference_block(res_plus, res_minus, res_plus_ref, res_minus_ref, title):
    xL = res_plus['x'] / cant.L
    z_plus = np.abs(res_plus['z']) * 1e12
    z_minus = np.abs(res_minus['z']) * 1e12
    zp_plus = np.abs(res_plus['zp'])
    zp_minus = np.abs(res_minus['zp'])
    z_plus_ref = np.abs(res_plus_ref['z']) * 1e12
    z_minus_ref = np.abs(res_minus_ref['z']) * 1e12
    zp_plus_ref = np.abs(res_plus_ref['zp'])
    zp_minus_ref = np.abs(res_minus_ref['zp'])

    dz = z_plus - z_minus
    dzp = zp_plus - zp_minus
    z_plus_n = z_plus / max(np.max(z_plus), 1e-30)
    z_minus_n = z_minus / max(np.max(z_minus), 1e-30)
    zp_plus_n = zp_plus / max(np.max(zp_plus), 1e-30)
    zp_minus_n = zp_minus / max(np.max(zp_minus), 1e-30)
    z_plus_ref_n = z_plus_ref / max(np.max(z_plus_ref), 1e-30)
    z_minus_ref_n = z_minus_ref / max(np.max(z_minus_ref), 1e-30)
    zp_plus_ref_n = zp_plus_ref / max(np.max(zp_plus_ref), 1e-30)
    zp_minus_ref_n = zp_minus_ref / max(np.max(zp_minus_ref), 1e-30)
    dz_n = z_plus_n - z_minus_n
    dzp_n = zp_plus_n - zp_minus_n

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

    axes[0, 0].plot(xL, z_plus, lw=2.2, color=plus_data['color'], label='+8 V DC')
    axes[0, 0].plot(xL, z_minus, lw=2.2, color=minus_data['color'], label='-8 V DC')
    axes[0, 0].plot(xL, z_plus_ref, lw=1.5, color=plus_data['color'], ls='--', alpha=0.8, label='+8 V DC @ 10 kHz')
    axes[0, 0].plot(xL, z_minus_ref, lw=1.5, color=minus_data['color'], ls='--', alpha=0.8, label='-8 V DC @ 10 kHz')
    axes[0, 1].plot(xL, dz, lw=2.2, color='black')
    axes[0, 1].axhline(0.0, color='0.5', ls='--', lw=1)

    axes[1, 0].plot(xL, zp_plus, lw=2.2, color=plus_data['color'], label='+8 V DC')
    axes[1, 0].plot(xL, zp_minus, lw=2.2, color=minus_data['color'], label='-8 V DC')
    axes[1, 0].plot(xL, zp_plus_ref, lw=1.5, color=plus_data['color'], ls='--', alpha=0.8, label='+8 V DC @ 10 kHz')
    axes[1, 0].plot(xL, zp_minus_ref, lw=1.5, color=minus_data['color'], ls='--', alpha=0.8, label='-8 V DC @ 10 kHz')
    axes[1, 1].plot(xL, dzp, lw=2.2, color='black')
    axes[1, 1].axhline(0.0, color='0.5', ls='--', lw=1)

    axes[0, 0].set_title('|z(x)| comparison')
    axes[0, 1].set_title('Delta|z| = |z|(+8V) - |z|(-8V)')
    axes[1, 0].set_title("|z'(x)| comparison")
    axes[1, 1].set_title("Delta|z'| = |z'|(+8V) - |z'|(-8V)")

    axes[0, 0].set_ylabel('|z| (pm)')
    axes[0, 1].set_ylabel('Delta|z| (pm)')
    axes[1, 0].set_ylabel("|z'| (dimensionless)")
    axes[1, 1].set_ylabel("Delta|z'|")

    for ax in axes.flat:
        ax.grid(True, alpha=0.25)
        ax.set_xlabel('Normalized position x/L')

    axes[0, 0].legend(loc='best')
    axes[1, 0].legend(loc='best')

    fig.suptitle(title, y=1.02)
    fig.tight_layout()

    fig_n, axes_n = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

    axes_n[0, 0].plot(xL, z_plus_n, lw=2.2, color=plus_data['color'], label='+8 V DC')
    axes_n[0, 0].plot(xL, z_minus_n, lw=2.2, color=minus_data['color'], label='-8 V DC')
    axes_n[0, 0].plot(xL, z_plus_ref_n, lw=1.5, color=plus_data['color'], ls='--', alpha=0.8, label='+8 V DC @ 10 kHz')
    axes_n[0, 0].plot(xL, z_minus_ref_n, lw=1.5, color=minus_data['color'], ls='--', alpha=0.8, label='-8 V DC @ 10 kHz')
    axes_n[0, 1].plot(xL, dz_n, lw=2.2, color='black')
    axes_n[0, 1].axhline(0.0, color='0.5', ls='--', lw=1)

    axes_n[1, 0].plot(xL, zp_plus_n, lw=2.2, color=plus_data['color'], label='+8 V DC')
    axes_n[1, 0].plot(xL, zp_minus_n, lw=2.2, color=minus_data['color'], label='-8 V DC')
    axes_n[1, 0].plot(xL, zp_plus_ref_n, lw=1.5, color=plus_data['color'], ls='--', alpha=0.8, label='+8 V DC @ 10 kHz')
    axes_n[1, 0].plot(xL, zp_minus_ref_n, lw=1.5, color=minus_data['color'], ls='--', alpha=0.8, label='-8 V DC @ 10 kHz')
    axes_n[1, 1].plot(xL, dzp_n, lw=2.2, color='black')
    axes_n[1, 1].axhline(0.0, color='0.5', ls='--', lw=1)

    axes_n[0, 0].set_title('Normalized |z(x)| comparison')
    axes_n[0, 1].set_title('Delta normalized |z|')
    axes_n[1, 0].set_title("Normalized |z'(x)| comparison")
    axes_n[1, 1].set_title("Delta normalized |z'|")

    axes_n[0, 0].set_ylabel('|z| / max|z|')
    axes_n[0, 1].set_ylabel('Delta normalized |z|')
    axes_n[1, 0].set_ylabel("|z'| / max|z'|")
    axes_n[1, 1].set_ylabel("Delta normalized |z'|")

    for ax in axes_n.flat:
        ax.grid(True, alpha=0.25)
        ax.set_xlabel('Normalized position x/L')

    axes_n[0, 0].legend(loc='best')
    axes_n[1, 0].legend(loc='best')
    fig_n.suptitle(title + ' (normalized)', y=1.02)
    fig_n.tight_layout()

    return (fig, axes), (fig_n, axes_n)

plot_shape_difference_block(
    res_plus_res,
    res_minus_res,
    res_plus_ref,
    res_minus_ref,
    title=f'Resonance shape comparison at f = {f_res_cmp/1e3:.3f} kHz',
)

plot_shape_difference_block(
    res_plus_ar,
    res_minus_ar,
    res_plus_ref,
    res_minus_ref,
    title=f'Antiresonance shape comparison at f = {f_ar_cmp/1e3:.3f} kHz',
)

print(f'+8 V resonance: {f_res_plus/1e3:.3f} kHz   -8 V resonance: {f_res_minus/1e3:.3f} kHz')
print(f'+8 V antiresonance: {f_ar_plus/1e3:.3f} kHz   -8 V antiresonance: {f_ar_minus/1e3:.3f} kHz')

## 4 - Null position as a function of contact stiffness

The local null positions are not fixed in general. In this section we sweep `k1` and track where the response minima occur for the `+8 V` and `-8 V` electrostatic cases.

- **Displacement null**: minimum of `|z(x)|` near the tip
- **Slope null**: minimum of `|z'(x)|` in the interior region around `x/L ~ 0.6`

This lets us see whether reversing the DC bias shifts the spatial structure of the mode.

Unlike the earlier version, this section tracks **explicit mode branches** (`mode 1` and `mode 2`) instead of whichever peak happens to be dominant. It also reports a displacement null only when a true **interior** minimum exists inside the near-tip search window.

In [ ]:
from eb_cr_afm.params import ContactParams

k1_sweep = np.logspace(0, 5, 17)
mode_indices = [0, 1]
branch_tracks = {}
null_track_cases = {
    'Pure mechanical': {
        'exc': make_excitation(dict(use_distributed_load=False, use_f0=False, bias_coupled_f0=False, f0=0.0, delta_V=0.0, V_AC=0.0)),
        'color': '#7f7f7f',
        'short_label': 'Mechanical',
    },
}
for name, data in profile_data.items():
    null_track_cases[name] = {
        'exc': data['exc'],
        'color': data['color'],
        'short_label': data['short_label'],
    }

def pick_interior_null(x_vals, amp_vals, lo, hi, require_interior=False):
    mask = (x_vals >= lo) & (x_vals <= hi)
    xm = x_vals[mask]
    am = amp_vals[mask]
    if len(xm) < 3:
        return np.nan
    idx = int(np.argmin(am))
    if require_interior and (idx == 0 or idx == len(xm) - 1):
        return np.nan
    return float(xm[idx])

for name, data in null_track_cases.items():
    branch_tracks[name] = {
        'color': data['color'],
        'short_label': data['short_label'],
        'k1': k1_sweep,
        'modes': {},
    }

    for mode_index in mode_indices:
        freqs = []
        disp_nulls = []
        slope_nulls = []

        for k1 in k1_sweep:
            c_use = ContactParams(
                k1=float(k1),
                k2=0.85 * float(k1),
                c1=contact.c1,
                c2=0.85 * contact.c1,
                psi=contact.psi,
            )

            sweep = frequency_sweep(cant, c_use, data['exc'], opts_base)
            modes = sho_analysis(sweep)
            if len(modes) <= mode_index:
                freqs.append(np.nan)
                disp_nulls.append(np.nan)
                slope_nulls.append(np.nan)
                continue

            f_res = float(modes[mode_index]['f_res'])
            opts_track = SolverOptions(
                n_points=2400,
                f_start=f_res - 1e3,
                f_stop=f_res + 1e3,
                f_step=250.0,
                g=opts_base.g,
                method='analytical',
            )
            res = solve_single_frequency(f_res, cant, c_use, data['exc'], opts_track)
            xL = res['x'] / cant.L
            z_amp = np.abs(res['z'])
            zp_amp = np.abs(res['zp'])

            disp_null = pick_interior_null(xL, z_amp, 0.92, 1.00, require_interior=True)
            slope_null = pick_interior_null(xL, zp_amp, 0.45, 0.70, require_interior=False)

            freqs.append(f_res)
            disp_nulls.append(disp_null)
            slope_nulls.append(slope_null)

        branch_tracks[name]['modes'][mode_index] = {
            'f_res': np.array(freqs, dtype=float),
            'disp_null_xL': np.array(disp_nulls, dtype=float),
            'slope_null_xL': np.array(slope_nulls, dtype=float),
        }

fig, axes = plt.subplots(3, 2, figsize=(13, 11), sharex='col')

for col, mode_index in enumerate(mode_indices):
    ax_f = axes[0, col]
    ax_d = axes[1, col]
    ax_s = axes[2, col]

    for name, track in branch_tracks.items():
        color = track['color']
        label = track['short_label']
        mode_track = track['modes'][mode_index]

        ax_f.plot(track['k1'], mode_track['f_res'] / 1e3, marker='o', lw=2, color=color, label=label)
        ax_d.plot(track['k1'], mode_track['disp_null_xL'], marker='o', lw=2, color=color, label=label)
        ax_s.plot(track['k1'], mode_track['slope_null_xL'], marker='o', lw=2, color=color, label=label)

    ax_f.set_title(f'Mode {mode_index + 1}')
    ax_f.set_ylabel('Branch frequency (kHz)')
    ax_d.set_ylabel('Displacement null x/L')
    ax_s.set_ylabel('Slope null x/L')
    ax_s.set_xlabel('Contact stiffness k1 (N/m)')

    for ax in (ax_f, ax_d, ax_s):
        ax.set_xscale('log')
        ax.grid(True, alpha=0.3)

axes[0, 0].legend(loc='best')
axes[0, 1].legend(loc='best')

fig.suptitle('Mode branches and branch-tracked null positions vs contact stiffness', y=1.02)
fig.tight_layout()

print('Branch-tracked null summary:')
for name, track in branch_tracks.items():
    for mode_index in mode_indices:
        mode_track = track['modes'][mode_index]
        finite_disp = mode_track['disp_null_xL'][np.isfinite(mode_track['disp_null_xL'])]
        finite_slope = mode_track['slope_null_xL'][np.isfinite(mode_track['slope_null_xL'])]
        disp_msg = 'no interior near-tip null' if len(finite_disp) == 0 else f"{finite_disp[0]:.4f} -> {finite_disp[-1]:.4f}"
        slope_msg = 'none' if len(finite_slope) == 0 else f"{finite_slope[0]:.4f} -> {finite_slope[-1]:.4f}"
        print(f"{track['short_label']:12s} mode {mode_index + 1}:  disp {disp_msg:>24s}   slope {slope_msg}")


## 5 - What to look for

- **Peak amplitude changes** tell you how strongly the electrostatic drive couples into the mode.
- **Normalized profile overlays** show whether electrostatics merely rescale the response or actually distort the shape.
- **Near-tip displacement zoom** is useful because the first contact-resonance displacement profile approaches a null at the tip.
- **Slope-null zoom near `x/L ~ 0.6`** is the most sensitive place to see small shape changes in the OBD-relevant profile.
- In the **2D spectra**, compare not only the bright resonance ridge but also how the low-amplitude troughs move and sharpen around the null regions.